In [0]:
%run /Workspace/Repos/Project_1705/Azure_Retail_Project/Project_Files/Functions/ACCESS_ADLS_GEN2_USING_SERVICE_PRINCIPALE 

In [0]:
from datetime import datetime
currentdate = datetime.now().strftime("%Y%m%d")
print(currentdate)

In [0]:
# currentdate = '20260524'

In [0]:
dbutils.widgets.text("read_container_name","silver","Read Container")
read_container_name = dbutils.widgets.get("read_container_name")

In [0]:
dbutils.widgets.text("write_container_name","gold","Write Container")
write_container_name = dbutils.widgets.get("write_container_name")

In [0]:
dbutils.widgets.dropdown("table_name","sellers",["customers","products","sellers"],"Table Name")
table_name = dbutils.widgets.get("table_name")

In [0]:
table_config = {

    "sellers": {
        "business_key": "seller_id"
    },

    "customers": {
        "business_key": "customer_id"
    },

    "products": {
        "business_key": "product_id"
    }

}

In [0]:
business_key = table_config[table_name]["business_key"]

In [0]:
read_base_path = f"abfss://{read_container_name}@retailstorage1881.dfs.core.windows.net"

In [0]:
write_base_path = f"abfss://{write_container_name}@retailstorage1881.dfs.core.windows.net"

In [0]:
from pyspark.sql.functions import current_date, lit
from delta.tables import DeltaTable
from py4j.protocol import Py4JJavaError

# Target Gold Path
target_path = f"{write_base_path}/{table_name}/"

# Read Silver Data
silver_df = spark.read.format('delta').load(f"{read_base_path}/{table_name}/{currentdate}/")
silver_df.display()

# All Source Columns
all_columns = silver_df.columns

# Compare Columns
compare_cols = [
    col for col in all_columns
    if col not in [
        business_key,
        "START_DATE",
        "END_DATE",
        "IS_CURRENT",
        "ingest_ts"
    ]
]

# Dynamic Compare Condition
compare_condition = " OR ".join(
    [
        f"COALESCE(CAST(TARGET.{col} AS STRING), '') <> COALESCE(CAST(SOURCE.{col} AS STRING), '')"

        for col in compare_cols
    ]
)

# Insert Columns
insert_cols = ", ".join(
    all_columns +
    [
        "START_DATE",
        "END_DATE",
        "IS_CURRENT"
    ]
)

# Source Columns
source_cols = ", ".join(
    [
        f"SOURCE.{col}"
        for col in all_columns
    ]
)

#Safe condition
try:
    dbutils.fs.ls(target_path + "/_delta_log")
    table_exists = True

except:
    table_exists = False

# FIRST RUN
# if not DeltaTable.isDeltaTable(spark, target_path):
if table_exists == False:

    print("First Run Started")

    initial_df = silver_df \
        .withColumn("START_DATE", current_date()) \
        .withColumn("END_DATE", lit(None).cast("date")) \
        .withColumn("IS_CURRENT", lit("Y"))

    initial_df.write \
        .format("delta") \
        .mode("overwrite") \
        .save(target_path)

    print("Initial Load Completed")


# FUTURE RUNS
else:
    print("Incremental SCD2 Load Started")
    silver_df.createOrReplaceTempView("SOURCE")

    # Expire Old Records
    merge_query = f"""
    MERGE INTO delta.`{target_path}` TARGET
    USING SOURCE
    ON TARGET.{business_key} = SOURCE.{business_key}
    AND TARGET.IS_CURRENT = 'Y'

    WHEN MATCHED AND ({compare_condition})

    THEN UPDATE SET
        TARGET.END_DATE = CURRENT_DATE(),
        TARGET.IS_CURRENT = 'N'
    """
    spark.sql(merge_query)

    # Insert New + Changed Records
    insert_query = f"""
    INSERT INTO delta.`{target_path}`
    ({insert_cols})
    SELECT {source_cols}, CURRENT_DATE(), NULL, 'Y'
    FROM SOURCE
    LEFT JOIN delta.`{target_path}` TARGET
    ON SOURCE.{business_key} = TARGET.{business_key}
    AND TARGET.IS_CURRENT = 'Y'
    WHERE TARGET.{business_key} IS NULL
    OR ({compare_condition})
    """
    spark.sql(insert_query)

    print("Incremental SCD2 Load Completed")

In [0]:
df1 = spark.read.format('delta').load(f"{write_base_path}/{table_name}/")
df1.display()